In [2]:
import pandas as pd
import glob
import os
import re

In [ ]:
import os
os.chdir(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))


In [ ]:
input_folder = 'data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/'
files = glob.glob(input_folder + '*.csv')

# Anchor to start and allow an optional numeric suffix like _1 or _2 after the material.
# This ensures we pick the material (letters+digits) at the beginning of the filename,
# ignoring an additional "_1" or "_2" piece before the wavelength.
pat = re.compile(
    r'^(?P<material>[A-Za-z0-9]+)(?:_\d+)?_(?P<exc>\d+nm)_(?P<loading>[\d\-]+)_g_L_(?P<sonication>\d+(?:\.\d+)?)min',
    re.IGNORECASE
)

# exclude files that contain "calculatedfrom" (case-insensitive)
file_list = [f for f in files if 'calculatedfrom' not in os.path.basename(f).lower()]

groups = {}
skipped = []

for f in file_list:
    name = os.path.basename(f)
    m = pat.search(name)
    if not m:
        skipped.append(name)
        continue
    material = m.group('material')
    # normed_material = norm_material(material)
    exc = m.group('exc')
    loading = m.group('loading')
    son = m.group('sonication')
    key = (material, exc, loading, son)
    groups.setdefault(key, []).append(f)

dfs_by_group = {}
for key, flist in groups.items():
    df_list = [pd.read_csv(fp) for fp in flist]
    dfs_by_group[key] = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

print(f"Found {len(groups)} groups, skipped {len(skipped)} files. Skipped examples: {skipped[:5]}")
for key, df in dfs_by_group.items():
    print(f"{key}: {len(groups[key])} files -> {df.shape}")

# Print the first element (key and dataframe) of the dict
if dfs_by_group:
    first_key = next(iter(dfs_by_group))
    print("\nFirst element key:", first_key)
    print("First element dataframe shape:", dfs_by_group[first_key].shape)
    print(dfs_by_group[first_key].head())
else:
    print("dfs_by_group is empty")

In [4]:
# filter existing dfs_by_group into two dictionaries for the requested loading/sonication combos
loading0_75_0min = {k: v for k, v in dfs_by_group.items() if k[2] == '0-75' and str(k[3]) == '0'}
loading0_75_10min = {k: v for k, v in dfs_by_group.items() if k[2] == '0-75' and str(k[3]) == '10'}

# summary
print(f"loading0-75_0min: {len(loading0_75_0min)} groups")
print(sorted(loading0_75_0min.keys()))
print(f"\nloading0-75_10min: {len(loading0_75_10min)} groups")
print(sorted(loading0_75_10min.keys()))

loading0-75_0min: 18 groups
[('BLPHI4', '365nm', '0-75', '0'), ('BLPHI4', '406nm', '0-75', '0'), ('BLPHI4', '450nm', '0-75', '0'), ('LN28', '406nm', '0-75', '0'), ('LN28', '450nm', '0-75', '0'), ('LN28SBA', '450nm', '0-75', '0'), ('LN31', '365nm', '0-75', '0'), ('LN31', '406nm', '0-75', '0'), ('LN31', '450nm', '0-75', '0'), ('LN33', '365nm', '0-75', '0'), ('LN33', '406nm', '0-75', '0'), ('LN33', '450nm', '0-75', '0'), ('LN46', '365nm', '0-75', '0'), ('LN46', '406nm', '0-75', '0'), ('LN46', '450nm', '0-75', '0'), ('LN55', '365nm', '0-75', '0'), ('LN55', '406nm', '0-75', '0'), ('LN55', '450nm', '0-75', '0')]

loading0-75_10min: 3 groups
[('LN28SBA', '450nm', '0-75', '10'), ('LN31', '450nm', '0-75', '10'), ('LN33', '450nm', '0-75', '10')]


## Produce bar plots for the single materials (Here the names are changes matching the ones in the publication)

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

d = loading0_75_0min

# def norm_material(name):
#     return 'BLPHI4' if name.upper().startswith('BLPHI') else name

material_label_map = {
    'BLPHI4': 'KPHI-lit',
    'LN55': 'KPHIS-1',
    'LN33': 'KPHIS-2',
    'LN31': 'KPHIS-3',
    'LN46': 'KPHI-b'
}

def exc_sort_key(exc):
    try:
        return int(re.search(r'(\d+)', exc).group(1))
    except Exception:
        return exc

def source_stems_for_key(key):
    return [os.path.splitext(os.path.basename(p))[0] for p in groups.get(key, [])]

mat_to_entries = {}
for key, df_ in d.items():
    material, exc, loading, son = key
    mat_to_entries.setdefault(material, []).append((key, exc, df_))

out_dir = "data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/comparison_plots/for_publication/"
os.makedirs(out_dir, exist_ok=True)

cmap = plt.get_cmap('tab10')

all_times_dir = os.path.join(out_dir, "all_times")
selected_times_dir = os.path.join(out_dir, "selected_times")
os.makedirs(all_times_dir, exist_ok=True)
os.makedirs(selected_times_dir, exist_ok=True)

def save_artifacts(df_out, meta, folder, csv_name, json_name):
    csv_path = os.path.join(folder, csv_name)
    json_path = os.path.join(folder, json_name)
    df_out.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return csv_path, json_path

for material, entries in mat_to_entries.items():
    entries = sorted(entries, key=lambda x: exc_sort_key(x[1]))
    excs = [exc for _, exc, _ in entries]
    if len(excs) != 3:
        continue

    display_mat = material_label_map.get(material, material)
    times_union = sorted({t for _, _, df_ in entries for t in df_['times'].tolist()})

    vals, errs = [], []
    combined_rows = []
    all_source_files, all_source_stems = set(), set()

    for key, exc, df_ in entries:
        grp = (
            df_.groupby('times', as_index=False)
               .agg({
                   'average_c': 'mean',
                   '95% confidence interval': 'mean',
                   'number of datapoints': 'first'
               })
               .sort_values('times')
        )
        grp['material'] = key[0]
        grp['material_display'] = display_mat
        grp['excitation'] = exc
        grp['loading'] = key[2]
        grp['sonication'] = key[3]
        grp['source_files'] = ';'.join(groups.get(key, []))
        grp['source_stems'] = ';'.join(source_stems_for_key(key))
        combined_rows.append(grp)

        all_source_files.update(groups.get(key, []))
        all_source_stems.update(source_stems_for_key(key))

        s = grp.set_index('times')['average_c'].reindex(times_union)
        e = grp.set_index('times')['95% confidence interval'].reindex(times_union)
        vals.append(s.fillna(0).values)
        errs.append(e.fillna(0).values)

    df_combined = pd.concat(combined_rows, ignore_index=True)

    x = np.arange(len(times_union))
    n = len(excs)
    bar_width = 0.8 / n
    offset = - (n - 1) * bar_width / 2

    plt.figure(figsize=(8, 4.5))
    for i, (exc, arr, err) in enumerate(zip(excs, vals, errs)):
        plt.bar(
            x + offset + i * bar_width, arr, width=bar_width, label=exc, color=cmap(i % 10),
            yerr=err, error_kw={'ecolor': 'black', 'elinewidth': 1, 'alpha': 0.8, 'capsize': 3}
        )

    plt.xticks(x, [int(t) if float(t).is_integer() else t for t in times_union])
    plt.xlabel('time / h')
    plt.ylabel(r'average $c$(H$_2$O$_2$) / mmol L$^{-1}$')
    #plt.title(f'{display_mat} — average_c by time for wavelengths ({", ".join(excs)})')
    plt.legend(title='excitation')
    #plt.grid(axis='y', linestyle=':', alpha=0.3)
    plt.tight_layout()

    img_fname = f"{display_mat}_0-75_g_L_0min_comparison.png"
    plt.savefig(os.path.join(out_dir, "all_times", img_fname), dpi=600)
    plt.close()

    meta_full = {
        "plot_type": "comparison_all_times",
        "material": material,
        "material_display": display_mat,
        "loading": "0-75",
        "sonication": "0",
        "excitations": excs,
        "times": [float(t) for t in times_union],
        "xlabel": "time / h",
        "ylabel": r'average $c$(H$_2$O$_2$) / mmol L$^{-1}$',
        "title": f'{display_mat} — average_c by time for wavelengths ({", ".join(excs)})',
        "legend_title": "excitation",
        "image_file": img_fname,
        "source_files": sorted(all_source_files),
        "source_stems": sorted(all_source_stems),
        "columns": list(df_combined.columns),
    }
    save_artifacts(
        df_combined,
        meta_full,
        all_times_dir,
        f"{display_mat}_0-75_g_L_0min_comparison.csv",
        f"{display_mat}_0-75_g_L_0min_comparison.json",
    )

    selected_times = [1.0, 2.0, 3.0, 4.0, 6.0]
    x_sel = np.arange(len(selected_times))
    vals_sel, errs_sel = [], []
    for key, exc, df_ in entries:
        grp = (
            df_.groupby('times', as_index=False)
               .agg({
                   'average_c': 'mean',
                   '95% confidence interval': 'mean',
                   'number of datapoints': 'first'
               })
               .sort_values('times')
        )
        s = grp.set_index('times')['average_c'].reindex(selected_times)
        e = grp.set_index('times')['95% confidence interval'].reindex(selected_times)
        vals_sel.append(s.fillna(0).values)
        errs_sel.append(e.fillna(0).values)

    plt.figure(figsize=(8, 4.5))
    for i, (exc, arr, err) in enumerate(zip(excs, vals_sel, errs_sel)):
        plt.bar(
            x_sel + offset + i * bar_width, arr, width=bar_width, label=exc, color=cmap(i % 10),
            yerr=err, error_kw={'ecolor': 'black', 'elinewidth': 1, 'alpha': 0.8, 'capsize': 3}
        )

    plt.xticks(x_sel, [int(t) for t in selected_times])
    plt.xlabel('time / h')
    plt.ylabel(r'average $c$(H$_2$O$_2$) / mmol L$^{-1}$')
    #plt.title(f'{display_mat} — average c(H$_2$O$_2$) / mmol L$^{-1}$')
    plt.tight_layout()

    sel_img_fname = f"{display_mat}_0-75_g_L_0min_comparison_selected_times.png"
    plt.savefig(os.path.join(out_dir, "selected_times", sel_img_fname), dpi=600)
    plt.close()

    df_selected = df_combined[df_combined['times'].isin(selected_times)].copy()
    meta_sel = {
        "plot_type": "comparison_selected_times",
        "material": material,
        "material_display": display_mat,
        "loading": "0-75",
        "sonication": "0",
        "selected_times": selected_times,
        "excitations": excs,
        "xlabel": "time / h",
        "ylabel": r'average $c$(H$_2$O$_2$) / mmol L$^{-1}$',
        "title": f'{display_mat} — average $c$(H$_2$O$_2$) / mmol L$^{-1}$',
        "image_file": sel_img_fname,
        "source_files": sorted(all_source_files),
        "source_stems": sorted(all_source_stems),
        "columns": list(df_selected.columns),
    }
    save_artifacts(
        df_selected,
        meta_sel,
        selected_times_dir,
        f"{display_mat}_0-75_g_L_0min_comparison_selected_times.csv",
        f"{display_mat}_0-75_g_L_0min_comparison_selected_times.json",
    )


# Making point plots for each wavelength

In [ ]:
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# point+line plots per wavelength for selected times
selected_times = [0.0, 1.0, 2.0, 3.0, 4.0, 6.0]

# materials to exclude
exclude_materials = {'LN28', 'LN28SBA', 'LN28_SBA'}

# Material name normalization
material_normalization = {'BLPHI': 'BLPHI4'}

# collect all wavelengths present, excluding LN28 / LN28_SBA (sorted by wavelength value)
all_excs = sorted({
    entry_exc
    for material, entry_list in mat_to_entries.items()
    if material not in exclude_materials
    for _, entry_exc, _ in entry_list
}, key=exc_sort_key)

# collect all materials present, excluding LN28 / LN28_SBA (with normalization)
all_materials = sorted({
    material_normalization.get(material, material)
    for material, entry_list in mat_to_entries.items()
    if material not in exclude_materials
})

out_dir = "data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/comparison_plots/for_publication/"
os.makedirs(out_dir, exist_ok=True)

# Fixed color per material (consistent across all wavelengths)
material_color_map = {
    'BLPHI4': '#FFA500',      # orange
    'LN55': '#FF0000',        # red
    'LN33': '#0000FF',        # blue
    'LN31': '#008000',        # green
    'LN46': '#000000'         # black
}

# Wavelength-appropriate colors for excitation legend
exc_color_map = {
    '365nm': '#9400D3',       # violet (UV)
    '406nm': '#0000FF',       # blue
    '450nm': '#00CED1'        # cyan (visible blue)
}

for exc_idx, exc in enumerate(all_excs):
    fig, ax = plt.subplots(figsize=(8, 4.5))

    material_handles = []
    material_labels = []
    rows_for_csv = []

    for i, (material, entry_list) in enumerate(mat_to_entries.items()):
        if material in exclude_materials:
            continue

        # Normalize material name
        normalized_material = material_normalization.get(material, material)
        display_mat = material_label_map.get(normalized_material, normalized_material)

        for _, entry_exc, grp in entry_list:
            if entry_exc != exc:
                continue

            y_s = (
                grp
                .groupby('times', as_index=True)['average_c']
                .mean()
                .reindex(selected_times)
            )
            e_s = (
                grp
                .groupby('times', as_index=True)['95% confidence interval']
                .mean()
                .reindex(selected_times)
            )

            y = y_s.fillna(0).values
            yerr = e_s.fillna(0).values

            # Use consistent material color
            mat_color = material_color_map.get(normalized_material, '#808080')
            line = ax.errorbar(
                selected_times, y, yerr=yerr,
                marker='o', linestyle='-',
                color=mat_color, capsize=3, linewidth=1.5
            )
            material_handles.append(line[0])
            material_labels.append(display_mat)

            tmp_df = pd.DataFrame({
                'material': normalized_material,
                'material_display': display_mat,
                'excitation': exc,
                'times': selected_times,
                'average_c': y,
                '95% confidence interval': yerr
            })
            rows_for_csv.append(tmp_df)

    # save dataframe used for this plot
    if rows_for_csv:
        plot_df = pd.concat(rows_for_csv, ignore_index=True)
        csv_name = f"materials_0-75_g_L_0min_{exc}_selected_times_lineplot_data.csv"
        plot_df.to_csv(os.path.join(out_dir, csv_name), index=False)

    ax.set_xticks([0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
    ax.set_xticklabels([int(t) if t == int(t) else t for t in [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]], fontname='Arial')
    #ax.set_xlabel(r'$\boldsymbol{time}\ \mathbf{/\ h}$', fontname='Arial', weight='bold')
    #ax.set_ylabel(r'$\mathbf{average}\ \boldsymbol{\mathit{c}}(\boldsymbol{\mathit{H}_2}\boldsymbol{\mathit{O}_2})\ /\ mmol\ L^{-1}$', fontname='Arial', weight='bold')
    ax.set_xlim(-0.1, max(selected_times) + 0.1)
    ax.set_ylim(-0.1, None)

    # show material legend only for the first plot
    if exc == '365nm':
        # Reorder handles and labels
        material_order = ['KPHI-lit', 'KPHI-b', 'KPHIS-1', 'KPHIS-2', 'KPHIS-3']
        ordered_handles = []
        ordered_labels = []
        for mat_label in material_order:
            for handle, label in zip(material_handles, material_labels):
                if label == mat_label:
                    ordered_handles.append(handle)
                    ordered_labels.append(label)
                    break
        
        leg1 = ax.legend(
            ordered_handles, ordered_labels,
            title='material',
            fontsize=14,
            title_fontsize=14,
            prop={'family': 'Arial', 'size': 14},
            loc='upper left'
        )
        ax.add_artist(leg1)

    # excitation legend for every plot with wavelength-appropriate color
    exc_color = exc_color_map.get(exc, '#000000')
    exc_proxy = Line2D([0], [0], color=exc_color, linestyle='-', linewidth=2, label=exc)
    ax.legend(
        handles=[exc_proxy],
        title='excitation wavelength',
        fontsize=12,
        title_fontsize=12,
        prop={'family': 'Arial'},
        loc='upper left',
        bbox_to_anchor=(0, 0.65) if exc == '365nm' else None
    )

    plt.tight_layout()

    fname = f"materials_0-75_g_L_0min_{exc}_selected_times_lineplot.png"
    save_path = os.path.join(out_dir, fname)
    plt.savefig(save_path, dpi=900)
    plt.close()
    print(f"Saved wavelength plot: {save_path}")
    if rows_for_csv:
        print(f"Saved plot dataframe: {os.path.join(out_dir, csv_name)}")

In [ ]:
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family': 'Arial',
    'font.sans-serif': ['Arial'],
})

# point+line plots per wavelength for selected times (10min sonication)
selected_times = [0.0, 1.0, 2.0, 3.0, 4.0, 6.0]

# materials to exclude
exclude_materials = {'LN28', 'LN28SBA', 'LN28_SBA'}

# Material name normalization
material_normalization = {'BLPHI': 'BLPHI4'}

# Use 10min sonication data
d_10min = loading0_75_10min

# Rebuild mat_to_entries for 10min data
mat_to_entries_10min = {}
for key, df_ in d_10min.items():
    material, exc, loading, son = key
    mat_to_entries_10min.setdefault(material, []).append((key, exc, df_))

# collect all wavelengths present, excluding LN28 / LN28_SBA (sorted by wavelength value)
all_excs_10min = sorted({
    entry_exc
    for material, entry_list in mat_to_entries_10min.items()
    if material not in exclude_materials
    for _, entry_exc, _ in entry_list
}, key=exc_sort_key)

out_dir_10min = "data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/comparison_plots/for_publication/10min_sonication/"
os.makedirs(out_dir_10min, exist_ok=True)

# Fixed color per material (consistent across all wavelengths)
material_color_map = {
    'BLPHI4': '#FFA500',      # orange
    'LN55': '#FF0000',        # red
    'LN33': '#0000FF',        # blue
    'LN31': '#008000',        # green
    'LN46': '#000000'         # black
}

# Wavelength-appropriate colors for excitation legend
exc_color_map = {
    '365nm': '#9400D3',       # violet (UV)
    '406nm': '#0000FF',       # blue
    '450nm': '#00CED1'        # cyan (visible blue)
}

for exc_idx, exc in enumerate(all_excs_10min):
    fig, ax = plt.subplots(figsize=(8, 4.5))

    material_handles = []
    material_labels = []
    rows_for_csv = []

    for i, (material, entry_list) in enumerate(mat_to_entries_10min.items()):
        if material in exclude_materials:
            continue

        # Normalize material name
        normalized_material = material_normalization.get(material, material)
        display_mat = material_label_map.get(normalized_material, normalized_material)

        for _, entry_exc, grp in entry_list:
            if entry_exc != exc:
                continue

            y_s = (
                grp
                .groupby('times', as_index=True)['average_c']
                .mean()
                .reindex(selected_times)
            )
            e_s = (
                grp
                .groupby('times', as_index=True)['95% confidence interval']
                .mean()
                .reindex(selected_times)
            )

            y = y_s.fillna(0).values
            yerr = e_s.fillna(0).values

            # Use consistent material color
            mat_color = material_color_map.get(normalized_material, '#808080')
            line = ax.errorbar(
                selected_times, y, yerr=yerr,
                marker='o', linestyle='-',
                color=mat_color, capsize=3, linewidth=1.5
            )
            material_handles.append(line[0])
            material_labels.append(display_mat)

            tmp_df = pd.DataFrame({
                'material': normalized_material,
                'material_display': display_mat,
                'excitation': exc,
                'times': selected_times,
                'average_c': y,
                '95% confidence interval': yerr
            })
            rows_for_csv.append(tmp_df)

    # save dataframe used for this plot
    if rows_for_csv:
        plot_df = pd.concat(rows_for_csv, ignore_index=True)
        csv_name = f"materials_0-75_g_L_10min_{exc}_data.csv"
        plot_df.to_csv(os.path.join(out_dir_10min, csv_name), index=False)

    ax.set_xticks(selected_times)
    ax.set_xticklabels([int(t) for t in selected_times], fontname='Arial')
    ax.set_xlabel(r'$time$ / h', fontname='Arial')
    ax.set_ylabel(r'$c(H_2O_2)$ / mmol L$^{-1}$', fontname='Arial')
    ax.set_xlim(-0.1, max(selected_times) + 0.1)
    ax.set_ylim(-0.1, None)

    # show material legend only for the first plot
    if exc == '450nm':
        leg1 = ax.legend(
            material_handles, material_labels,
            title='material',
            fontsize=9,
            title_fontsize=10,
            prop={'family': 'Arial'},
            loc='upper left'
        )
        ax.add_artist(leg1)

    # excitation legend for every plot with wavelength-appropriate color
    exc_color = exc_color_map.get(exc, '#000000')
    exc_proxy = Line2D([0], [0], color=exc_color, linestyle='-', linewidth=2, label=exc)
    ax.legend(
        handles=[exc_proxy],
        title='excitation wavelength',
        fontsize=9,
        title_fontsize=10,
        prop={'family': 'Arial'},
        loc='upper left',
        bbox_to_anchor=(0, 0.65) if exc == '450nm' else None
    )

    plt.tight_layout()

    fname = f"materials_0-75_g_L_10min_{exc}_data.png"
    plt.savefig(os.path.join(out_dir_10min, fname), dpi=600)
    plt.close()
    print(f"Saved wavelength plot (10min sonication): {os.path.join(out_dir_10min, fname)}")
    if rows_for_csv:
        print(f"Saved plot dataframe (10min sonication): {os.path.join(out_dir_10min, csv_name)}")

## Here, the apparent quantum efficiency (AQE) is calculated

In [ ]:
from astropy import constants as const
from astropy import units as u

# assumptions/constants for AQY calculation

# assumptions:
# - reaction volume V is constant: 20 mL
# - LED power density: 50 mW / cm^2
# - illuminated area 3.5 cm^2 (so P = 50 mW)
V = 20 * u.ml
# P_density = 175 * u.mW / (u.cm ** 2)
irradiation_diameter = 3.5 * u.cm # this is the diameter of the illuminated reactors
irradiated_area = np.pi * (irradiation_diameter / 2) ** 2
irradiated_area = irradiated_area.to(u.cm**2)
P_density = 50 * u.mW / (u.cm ** 2)
#irradiated_area = 4 * u.cm**2 # this is the diameter of the illuminated reactors
irradiation_P = (P_density * irradiated_area).to(u.W)  # total power in W
print(f"Irradiation power: {irradiation_P:.2f}")

out_dir = "data/evaluation/material_excwavelength_loading_sonication/ignore_synthesis_date/comparison_plots/for_publication/"
os.makedirs(out_dir, exist_ok=True)

rows = []
for (material, exc, loading, son), df in dfs_by_group.items():
    if loading != '0-75' or str(son) != '0':
        continue
    if df.empty:
        continue

    # normalize material: BLPHI and BLPHI4 -> BLPHI4
    norm_material = 'BLPHI4' if str(material).upper().startswith('BLPHI') else material

    # aggregate by time so index is unique (mean for average_c and ci; sum for number of datapoints)
    agg = df.groupby('times', as_index=True).agg({
        'average_c': 'mean',
        '95% confidence interval': 'mean',
        'standard error': 'mean',
        'number of datapoints': 'sum'
    })

    # try exact 4.0, otherwise nearest time (if present)
    target = 4.0
    matches = np.isclose(agg.index.values.astype(float), target)
    if not matches.any():
        # skip if no 4h measurement
        continue
    row = agg.iloc[np.where(matches)[0][0]]

    rows.append({
        'material': norm_material,
        'excitation': exc,
        'loading': loading,
        'sonication_min': son,
        'times': target,
        'average_c': row['average_c'],
        '95%_ci': row['95% confidence interval'],
        'std_err': row['standard error'],
        'n_points': int(row['number of datapoints'])
    })
    
big_df = pd.DataFrame(rows)

# identifier column requested
big_df['id'] = big_df.apply(lambda r: f"{r['material']}_{r['excitation']}_{r['loading']}_g-L_{r['sonication_min']}min", axis=1)
# display name mapping (use existing material_label_map)
def map_display(name):
    key = str(name)
    return material_label_map.get(key, key)

big_df['material_display'] = big_df['material'].apply(map_display)
big_df['id_display'] = big_df.apply(lambda r: f"{r['material_display']}_{r['excitation']}_{r['loading']}_g-L_{r['sonication_min']}min", axis=1)
print("Big dataframe shape:", big_df.shape)

def compute_aqy_for_row(r):
    try:
        conc = r['average_c'] * u.mmol / u.L          # concentration given as mmol L^-1
        moles = (conc * V).to(u.mol)                 # moles H2O2 produced
        n_molecules = (moles * const.N_A).to(u.dimensionless_unscaled)
        #print(f"n_molecules: {n_molecules}") #activate for checking the number of H2O2 molecules

        # reaction time (times column is in hours)
        t = (float(r['times']) * u.hour).to(u.s)
        if t.value <= 0:
            return np.nan

        # parse excitation wavelength string like '450nm'
        wl_str = str(r['excitation']).strip().lower()
        if wl_str.endswith('nm'):
            wl = float(wl_str.replace('nm', '')) * u.nm
        else:
            # if unparsable, return NaN
            return np.nan

        # number of incident photons = P * time / (h*c / lambda) = P * time * lambda / (h*c)
        photon_energy = (const.h * const.c / wl).to(u.J)
        n_photons = (irradiation_P * t) / photon_energy

        aqy = (2* n_molecules / n_photons).decompose().value
        return aqy
    except Exception:
        return np.nan

big_df['AQY'] = big_df.apply(compute_aqy_for_row, axis=1)
big_df['AQY_percent'] = big_df['AQY'] * 100.0
print(big_df[['material', 'excitation', 'average_c', 'AQY', 'AQY_percent']])
# save big dataframe (now including AQY)
big_df.to_csv(os.path.join(out_dir, "all_materials_4h_0min_summary.csv"), index=False)

# Create second plot excluding LN28, LN28SBA
big_df_filtered = big_df[~big_df['material'].isin(['LN28', 'LN28SBA'])].copy()
print(f"Filtered big dataframe shape (excluding LN28, LN28SBA): {big_df_filtered.shape}")

# Create pivot tables for the filtered data
pivot_val_filtered = big_df_filtered.pivot_table(
    index='material_display', 
    columns='excitation', 
    values='AQY_percent', 
    aggfunc='mean'
)

pivot_err_filtered = big_df_filtered.pivot_table(
    index='material_display', 
    columns='excitation', 
    values='95%_ci', 
    aggfunc='mean'
)

# Convert error from concentration to AQY_percent proportionally
# This is an approximation - multiply by the ratio of AQY_percent to average_c
for idx in pivot_err_filtered.index:
    for col in pivot_err_filtered.columns:
        if pd.notna(pivot_err_filtered.loc[idx, col]):
            # Find corresponding row in big_df_filtered
            mask = (big_df_filtered['material_display'] == idx) & (big_df_filtered['excitation'] == col)
            if mask.any():
                avg_c = big_df_filtered.loc[mask, 'average_c'].values[0]
                aqy_pct = big_df_filtered.loc[mask, 'AQY_percent'].values[0]
                ci = big_df_filtered.loc[mask, '95%_ci'].values[0]
                if avg_c > 0:
                    pivot_err_filtered.loc[idx, col] = ci * (aqy_pct / avg_c)

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

x_pos = np.arange(len(pivot_val_filtered.index))
width = 0.25
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, exc in enumerate(['365nm', '406nm', '450nm']):
    if exc in pivot_val_filtered.columns:
        vals = pivot_val_filtered[exc].values
        errs = pivot_err_filtered[exc].values
        ax.bar(x_pos + i * width, vals, width, 
               yerr=errs, 
               label=exc.replace('nm', ' nm'), 
               color=colors[i],
               capsize=5)

#ax.set_xlabel('Material', fontsize=12)
ax.set_ylabel('AQY (%)', fontsize=12, weight='bold', fontname='Arial')
#ax.set_title('Apparent Quantum Efficiency at 4h (LN28 excluded)', fontsize=14)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(pivot_val_filtered.index, rotation=45, ha='right')
ax.legend(title='excitation wavelength')

plt.tight_layout()
save_path_aqy = os.path.join(out_dir, 'summary_AQE_LN28_excluded.png')
plt.savefig(save_path_aqy, dpi=900, bbox_inches='tight')
print(f"Saved AQY plot to: {save_path_aqy}")
plt.show()

## Evaluating the kinetics of the reaction

In [37]:
from scipy.optimize import curve_fit
from pathlib import Path

# output folder
kin_outdir = Path(out_dir) / "kinetic_fitting_order_scan"
kin_outdir.mkdir(parents=True, exist_ok=True)

def normalize_material(mat):
    # requested: treat BLPHI as BLPHI4
    return "BLPHI4" if mat == "BLPHI" else mat

# Build long dataframe directly from existing grouped data (0-75 g/L, 0 min)
rows = []
for (material, exc, loading, son), df_ in loading0_75_0min.items():
    tmp = df_[['times', 'average_c', '95% confidence interval', 'standard error', 'number of datapoints']].copy()
    tmp = tmp[tmp['times'] <= 6].copy()  # requested: only datapoints within 6 h
    if tmp.empty:
        continue
    tmp['material'] = normalize_material(material)
    tmp['excitation'] = exc
    rows.append(tmp)

kin_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

def nth_order_growth(t, k, c_max, n):
    t = np.asarray(t, dtype=float)
    if np.isclose(n, 1.0):
        return c_max * (1.0 - np.exp(-k * t))
    base = 1.0 + (n - 1.0) * k * (c_max ** (n - 1.0)) * t
    base = np.maximum(base, 1e-12)
    return c_max * (1.0 - base ** (-1.0 / (n - 1.0)))

def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return np.nan if ss_tot == 0 else 1 - (ss_res / ss_tot)

n_grid = np.linspace(0, 4, 81)
best_rows = []

for (material, exc), sub in kin_df.groupby(['material', 'excitation']):
    sub = sub.groupby('times', as_index=False)['average_c'].mean().sort_values('times')
    sub = sub[sub['times'] <= 6].copy()  # enforce 6 h limit again after grouping

    if len(sub) < 3:
        continue

    # fit growth phase (within <= 6 h)
    t_peak = sub.loc[sub['average_c'].idxmax(), 'times']
    fit_sub = sub[sub['times'] <= t_peak].copy()

    if len(fit_sub) < 3:
        continue

    x = fit_sub['times'].values.astype(float)
    y = fit_sub['average_c'].values.astype(float)

    best = None
    c_guess = max(y.max() * 1.2, 1e-6)
    k_guess = 0.1

    for n in n_grid:
        try:
            popt, _ = curve_fit(
                lambda t, k, c_max: nth_order_growth(t, k, c_max, n),
                x, y,
                p0=[k_guess, c_guess],
                bounds=(0, np.inf),
                maxfev=20000
            )
            k_fit, c_max_fit = popt
            y_hat = nth_order_growth(x, k_fit, c_max_fit, n)
            r2 = r_squared(y, y_hat)

            if (best is None) or (r2 > best['r2']):
                best = {'n': n, 'k': k_fit, 'c_max': c_max_fit, 'r2': r2}
        except Exception:
            pass

    if best is None:
        continue

    # Plot raw data + best fit (<= 6 h only)
    x_all = sub['times'].values.astype(float)
    y_all = sub['average_c'].values.astype(float)
    x_fit = np.linspace(x.min(), x.max(), 300)
    y_fit = nth_order_growth(x_fit, best['k'], best['c_max'], best['n'])

    disp_mat = material_label_map.get(material, material) if 'material_label_map' in globals() else material

    plt.figure(figsize=(6.5, 4.2))
    plt.plot(x_all, y_all, 'o', color='black', label='data (<=6 h)')
    plt.plot(x_fit, y_fit, '-', color='tab:blue',
             label=f"best fit: n={best['n']:.2f}, R²={best['r2']:.4f}")
    plt.xlabel('time / h')
    plt.ylabel(r'average c / mmol L$^{-1}$')
    plt.title(f'{disp_mat} @ {exc}')
    plt.grid(alpha=0.3, linestyle=':')
    plt.legend()
    plt.tight_layout()

    safe_disp = str(disp_mat).replace(" ", "_")
    fname = f"{safe_disp}_{exc}_best_order_fit_upto6h.png"
    plt.savefig(kin_outdir / fname, dpi=300, bbox_inches='tight')
    plt.close()

    best_rows.append({
        'material': material,                   # normalized key (BLPHI merged into BLPHI4)
        'material_display': disp_mat,           # mapped display name
        'excitation': exc,
        'best_order_n': best['n'],
        'k': best['k'],
        'c_max': best['c_max'],
        'r_squared': best['r2'],
        'n_points_fit': len(fit_sub),
        't_peak_used_h': t_peak
    })

best_fit_df = pd.DataFrame(best_rows)
if not best_fit_df.empty:
    best_fit_df = best_fit_df.sort_values(['material', 'excitation'])

summary_path = kin_outdir / "best_fit_summary.csv"
best_fit_df.to_csv(summary_path, index=False)

print(best_fit_df)
print(f"\nSaved plots and summary to: {kin_outdir}")


   material material_display excitation  best_order_n             k  \
0    BLPHI4         KPHI-lit      365nm          0.85  2.885598e-01   
1    BLPHI4         KPHI-lit      406nm          0.15  1.321718e+00   
2    BLPHI4         KPHI-lit      450nm          0.30  1.832698e+00   
3      LN28             LN28      406nm          0.55  6.197009e-01   
4      LN28             LN28      450nm          0.10  2.989995e+00   
5   LN28SBA          LN28SBA      450nm          0.05  1.075581e+00   
6      LN31          KPHIS-3      365nm          0.00  4.161417e+00   
7      LN31          KPHIS-3      406nm          0.15  2.757310e+00   
8      LN31          KPHIS-3      450nm          0.40  1.577369e+00   
9      LN33          KPHIS-2      365nm          0.00  5.103028e+00   
10     LN33          KPHIS-2      406nm          0.00  3.136854e+00   
11     LN33          KPHIS-2      450nm          0.35  2.086591e+00   
12     LN46           KPHI-b      365nm          0.05  1.932369e+00   
13    